In [31]:
from pathlib import Path
import numpy as np
import pandas as pd

In [32]:
# compute ratio-based multimodal combination metrics (per-story profiles)
# mci_1 = tanh(MCC / (GrooVist + ε))
# mci_2 = tanh(GrooVist / (MCC + ε))
# save both variants for zero and nan policies

EPSILON = 1e-8
gv_dir = Path('./analysis_data/groovist')
mcc_dir = Path('./analysis_data/mcc')
out_dir = Path('./analysis_data/mci')
out_dir.mkdir(parents=True, exist_ok=True)

In [33]:
# build per-story mci tables for one policy (no saving here)
df_gv = pd.read_csv(gv_dir / 'groovist_metrics_per_story.csv')
df_gv = df_gv[['model', 'prompt', 'story_id', 'groovist']].copy()
df_gv['story_id'] = df_gv['story_id'].astype(int)

def build_mci_for_policy(policy):
    # MCC is exported per-character; derive per-story MCC by averaging characters within each story
    df_mcc_char = pd.read_csv(mcc_dir / f'mcc_metrics_per_character_{policy}_policy.csv')
    df_mcc_char = df_mcc_char[['model', 'prompt', 'story_id', 'MCC_char']].copy()
    df_mcc_char['story_id'] = df_mcc_char['story_id'].astype(int)

    df_mcc_story = (
        df_mcc_char.groupby(['model', 'prompt', 'story_id'])['MCC_char']
        .mean()
        .reset_index()
        .rename(columns={'MCC_char': 'MCC'})
    )

    df_mci = df_gv.merge(df_mcc_story, on=['model', 'prompt', 'story_id'], how='left')

    # ratio variants
    df_mci['mci_tanh_mcc_over_gv'] = np.tanh(df_mci['MCC'] / (df_mci['groovist'] + EPSILON))
    df_mci['mci_tanh_gv_over_mcc'] = np.tanh(df_mci['groovist'] / (df_mci['MCC'] + EPSILON))

    # optional symmetric summary
    #df_mci['mci_ratio_mean'] = (df_mci['mci_tanh_mcc_over_gv'] + df_mci['mci_tanh_gv_over_mcc']) / 2.0

    mci_agg = (
        df_mci.groupby(['model', 'prompt'])
        .agg(
            groovist_mean=('groovist', 'mean'),
            MCC_mean=('MCC', 'mean'),
            mci_tanh_mcc_over_gv_mean=('mci_tanh_mcc_over_gv', 'mean'),
            mci_tanh_gv_over_mcc_mean=('mci_tanh_gv_over_mcc', 'mean'),
            #mci_ratio_mean_mean=('mci_ratio_mean', 'mean'),
            observed_rows=('mci_tanh_mcc_over_gv', 'count'),
            total_rows=('mci_tanh_mcc_over_gv', 'size')
        )
        .reset_index()
        .assign(missing_policy=policy, unit='story')
    )

    return df_mci, mci_agg

In [34]:
df_mci_zero, mci_agg_zero = build_mci_for_policy('zero')
mci_agg_zero

,model,prompt,groovist_mean,MCC_mean,mci_tanh_mcc_over_gv_mean,mci_tanh_gv_over_mcc_mean,observed_rows,total_rows,missing_policy,unit
0,claude45,large,0.732562,0.414883,0.467782,0.892638,60,60,zero,story
1,claude45,original,0.748818,0.407661,0.457893,0.903083,60,60,zero,story
2,gpt4o,large,0.562436,0.471143,0.595880,0.783314,60,60,zero,story
3,gpt4o,original,0.559174,0.447196,0.576091,0.800745,60,60,zero,story
4,human,large,0.584672,0.385740,0.529435,0.841881,60,60,zero,story
5,human,original,0.454289,0.378001,0.447202,0.674735,60,60,zero,story
6,internvl3,large,0.584757,0.395359,0.517178,0.842424,60,60,zero,story
7,internvl3,original,0.595736,0.406159,0.531409,0.836080,60,60,zero,story
8,llama4scout,large,0.310965,0.225111,0.341540,0.612893,60,60,zero,story
9,llama4scout,original,0.352517,0.095177,0.148437,0.828541,60,60,zero,story


In [35]:
# zero-policy interpretation notes
# this table includes missing stories with mcc-side fill (0), so ratios are more conservative
# compare mci_tanh_mcc_over_gv_mean vs mci_tanh_gv_over_mcc_mean for asymmetry direction
# mci_ratio_mean_mean is the symmetric summary of both variants

# groovist: is the story about what is visually present? grounding quality, takes into account temporal misalignment, so if something is mentioned in text 4, but shown in image 1,  the metric is designed to capture that for the purposes of visual grounding; the idea is to capture when things that are mentioned are visually present
# mcc: do text-side and image-side character continuity match over the sequence?
# a story can be well grounded (groovist high), but still temporally misaligned with characters (lower mcc)
# groovist looks at all noun phrases, incl characters; mcc focuses on characters only

# humans and models seem to be generally better at mentioning what's in the images rather than matching character continuity between the modalities; grounding at noun/object level is easier (makes sense)
# given that groovist is high, we will rely on tanh(mcc/groovist) -> given grounding quality, how strong is continuity alignment?

In [36]:
# overall, humans don't seem to have such high grounding into images as models do (llama4 is an outlier)
# similarly, the same is observed for humans in mcc
# for mcc, again, humans are not as high as models
# humans seem to have a relative balance between grounding and character continuity, is this preserved in models?

# in humans: show some grounding and continuity, not extreme mismatch
# claude and qwen are closes to humans, but not all models match the balance in humans...

In [37]:
df_mci_nan, mci_agg_nan = build_mci_for_policy('nan')
mci_agg_nan

,model,prompt,groovist_mean,MCC_mean,mci_tanh_mcc_over_gv_mean,mci_tanh_gv_over_mcc_mean,observed_rows,total_rows,missing_policy,unit
0,claude45,large,0.732562,0.569161,0.629318,0.844435,47,60,nan,story
1,claude45,original,0.748818,0.561781,0.618824,0.856823,47,60,nan,story
2,gpt4o,large,0.562436,0.638426,0.791947,0.703286,47,60,nan,story
3,gpt4o,original,0.559174,0.598851,0.762191,0.730166,47,60,nan,story
4,human,large,0.584672,0.528356,0.710912,0.778698,47,60,nan,story
5,human,original,0.454289,0.530932,0.611860,0.644924,47,60,nan,story
6,internvl3,large,0.584757,0.525183,0.683440,0.787060,47,60,nan,story
7,internvl3,original,0.595736,0.541210,0.698415,0.773445,47,60,nan,story
8,llama4scout,large,0.310965,0.295860,0.436438,0.547529,47,60,nan,story
9,llama4scout,original,0.352517,0.122853,0.191170,0.823560,47,60,nan,story


In [38]:
# nan-policy interpretation notes
# this table excludes missing mcc-side matches from the mcc mean/ratio means
# means are usually higher than zero-policy because zero-fill rows are removed
# compare against zero-policy to quantify sensitivity to missingness handling

In [39]:
# save outputs only after reviewing zero/nan aggregate tables and notes
for policy, df_mci_policy, mci_agg_policy in [
    ('zero', df_mci_zero, mci_agg_zero),
    ('nan', df_mci_nan, mci_agg_nan),
]:
    df_mci_policy.to_csv(out_dir / f'mci_ratio_metrics_per_story_{policy}_policy.csv', index=False)
    mci_agg_policy.to_csv(out_dir / f'mci_ratio_metrics_agg_{policy}_policy.csv', index=False)

print('Saved MCI ratio outputs for policies: zero, nan')

Saved MCI ratio outputs for policies: zero, nan


In [40]:
neg = df_mci_zero[df_mci_zero['mci_tanh_mcc_over_gv'] < 0].copy()
print('negative rows:', len(neg))
display(neg[['model','prompt','story_id','groovist','MCC','mci_tanh_mcc_over_gv']])
print('Any negative MCC?', (df_mci_zero['MCC'] < 0).any())
print('All negative rows have negative groovist?', (neg['groovist'] < 0).all())

negative rows: 5


,model,prompt,story_id,groovist,MCC,mci_tanh_mcc_over_gv
25,human,original,7065,-0.111006,0.712815,-0.999995
26,human,original,6921,-0.055913,0.698150,-1.000000
31,human,original,6111,-0.048480,0.507729,-1.000000
586,llama4scout,large,3230,-0.131174,0.115529,-0.706787
592,llama4scout,large,13596,-0.051533,0.761594,-1.000000


Any negative MCC? False
All negative rows have negative groovist? True


In [ ]:
# note on negative mci_tanh_mcc_over_gv values:
# they are expected with this formula, not a bug
# mci_tanh_mcc_over_gv = tanh(MCC / (groovist + ε))
# MCC is non-negative here, so negatives occur when groovist < 0
# interpretation: sign-sensitive ratio marks anti-alignment in the grounding term for those stories